In [ ]:
import tensorflow as tf
import keras
import numpy as np
import math
import pandas as pd
from collections import OrderedDict
from keras import layers
from sklearn.model_selection import KFold
import scipy
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import json
import random
import ot
import pickle
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import pandas as pd
from sklearn.model_selection import KFold
from scipy import stats
import matplotlib.pyplot as plt

# 1.PAL-AI

In [ ]:
output_dir = "/public/home/zhangyt/project/ai/PAL-AI"

In [ ]:
def custom_metric(y_true, y_pred):
    y_true_mean = keras.backend.mean(y_true, axis=0)
    y_pred_mean = keras.backend.mean(y_pred, axis=0)
    r_numerator = keras.backend.sum((y_true - y_true_mean) * (y_pred - y_pred_mean))
    x_square_sum = keras.backend.sum((y_true - y_true_mean) * (y_true - y_true_mean))
    y_square_sum = keras.backend.sum((y_pred - y_pred_mean) * (y_pred - y_pred_mean))
    r_denominator = keras.backend.sqrt(x_square_sum * y_square_sum)
    r = r_numerator / r_denominator
    metric_out = -(keras.backend.mean(r) ** 2)
    return metric_out

In [ ]:
def pal_model(
    filters: int = 64,
    kernel_size: int = 5,
    dropout_rate: float = 0.1,
    learning_rate: float = 0.001,
    l1: float = 0.01,
    l2: float = 0.000001,
) -> keras.Model:
    model = keras.Sequential(
        [
            layers.Input(shape=(1056, 4)),
            layers.Conv1D(filters=filters, kernel_size=kernel_size),
            layers.LayerNormalization(),
            layers.Activation(keras.activations.selu),
            layers.Conv1D(filters=filters, kernel_size=kernel_size),
            layers.Dropout(rate=dropout_rate),
            layers.MaxPooling1D(pool_size=2),
            layers.LayerNormalization(),
            layers.Activation(keras.activations.selu),
            layers.GRU(
                units=96,
                go_backwards=False,
                kernel_regularizer=keras.regularizers.l1_l2(l1, l2),
            ),
            layers.BatchNormalization(),
            layers.Activation(keras.activations.selu),
            layers.Dense(
                units=96,
                kernel_regularizer=keras.regularizers.l1_l2(l1, l2),
            ),
            layers.Dropout(rate=dropout_rate),
            layers.BatchNormalization(),
            layers.Activation(keras.activations.selu),
            layers.Dense(units=1),
        ]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae", custom_metric],
    )
    return model

In [ ]:
pal_model().summary()

In [ ]:
def train_model(x_train, y_train, x_test, y_test, session_id: str = "", **kwargs):
    model = pal_model(**kwargs)
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=output_dir + "/models/" + session_id + "_callback_best_model.h5",
            monitor="val_loss",
            mode="min",
            save_best_only=True,
        ),
        keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10),
    ]
    out = model.fit(
        x=x_train,
        y=y_train,
        batch_size=50,
        epochs=30,
        verbose=1,
        shuffle=True,
        callbacks=callbacks,
        validation_data=(x_test, y_test),
    )
    model = keras.models.load_model(
        output_dir + "/models/" + session_id + "_callback_best_model.h5",
        custom_objects={"custom_metric": custom_metric},
    )
    return out, model

In [ ]:
y_dict = {}
with open(
    "/public/home/zhangyt/project/ai/PAL-AI/data/XL_oo_pp_7h_tail_length_change.txt",
    "r",
) as f:
    f.readline()
    for i, line in enumerate(f):
        temp = line.strip().split("\t")
        if int(temp[2]) >= 50:
            y_dict[temp[0]] = float(temp[1])

In [ ]:
utr_ls = []
with open("/public/home/zhangyt/project/ai/PAL-AI/data/XL_utr3.fa", "r") as f:
    fasta = f.readlines()
    for i in range(int(len(fasta) / 2)):
        id = fasta[i * 2].rstrip().lstrip(">")
        seq = fasta[i * 2 + 1]
        if ("uncertain" in id) or (len(seq) < 10):
            continue
        else:
            id_parse = id.split("__")
            seq = seq.upper()
            if len(seq) <= 1053:
                norm_seq = "N" * (1053 - len(seq)) + seq + "AAA"
            else:
                norm_seq = seq[-1053:] + "AAA"
            utr_ls.append([id_parse[0], norm_seq])

In [ ]:
def encode_seq(seq: str) -> np.ndarray:
    """convert a DNA sequence to one-hot encoding matrix

    Args:
        seq (str): DNA sequence to encode

    Returns:
        np.ndarray: one-hot encoding matrix[A,C,G,T], N is [0,0,0,0]
    """
    mat = np.zeros((len(seq), 4))
    for i in range(len(seq)):
        if seq[i] == "A":
            mat[i, 0] = 1.0
        elif seq[i] == "C":
            mat[i, 1] = 1.0
        elif seq[i] == "G":
            mat[i, 2] = 1.0
        elif seq[i] == "T" or seq[i] == "U":
            mat[i, 3] = 1.0
    return mat

In [ ]:
x_encode_ls = []
id_ls = []
y_polya_ls = []
for id, seq in utr_ls:
    try:
        y_polya_ls.append(y_dict[id])
        id_ls.append(id)
        x_encode_ls.append(encode_seq(seq))
    except KeyError:
        continue
x_ary = np.stack(x_encode_ls, axis=0)
y_df = pd.DataFrame({"id": id_ls, "y": y_polya_ls})

In [ ]:
np.save("/public/home/zhangyt/project/ai/PAL-AI/data/x_input.npy", x_ary)
y_df.to_csv("/public/home/zhangyt/project/ai/PAL-AI/data/y_input.csv", index=False)

In [ ]:
x_ary = np.load("/public/home/zhangyt/project/ai/PAL-AI/data/x_input.npy")
y_df = pd.read_csv("/public/home/zhangyt/project/ai/PAL-AI/data/y_input.csv")

In [ ]:
print(x_ary.shape, y_df.shape)

In [ ]:
def loss_plot(train, test, fn):
    plt.clf()
    plt.plot(train, label="train")
    plt.plot(test, label="test")
    plt.legend()
    plt.xlabel("Epochs")
    plt.ylabel("Loss or metric")
    plt.savefig(fn)

In [ ]:
def scatter_plot(x, y, fn):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )
    plt.savefig(fn, dpi=300, bbox_inches="tight")

In [ ]:
stat = pd.DataFrame(
    None, columns=["group", "rep", "r2", "loss", "mae", "custom_metric"]
)
res = pd.DataFrame(None, columns=["group", "id", "y", "y_pred"])
kf = KFold(n_splits=10, shuffle=True, random_state=57)
for i, (train_idx, test_idx) in enumerate(kf.split(x_ary, y_df["y"])):
    best_r2 = 0
    for j in range(5):
        x_train = x_ary[train_idx]
        x_test = x_ary[test_idx]
        y_train = y_df.loc[train_idx, "y"].to_numpy()
        y_test = y_df.loc[test_idx, "y"].to_numpy()
        history, model = train_model(
            x_train, y_train, x_test, y_test, session_id="XL_oo_pp"
        )
        model.save(
            output_dir
            + "/models/"
            + "XL_oo_pp"
            + "_best_model_CV_group_"
            + str(i + 1)
            + "_rep_"
            + str(j + 1)
            + ".h5"
        )
        scores = model.evaluate(x_test, y_test, batch_size=50, verbose=1)
        y_pred = model.predict(x_test, verbose=1).ravel()
        r_value = stats.linregress(y_test, y_pred).rvalue
        if r_value**2 > best_r2:
            best_r2 = r_value**2
            best_model = model
        del model
        keras.backend.clear_session()
        stat.loc[len(stat)] = [i + 1, j + 1, round(r_value**2, 3)] + list(scores)
        loss_plot(
            train=history.history["loss"],
            test=history.history["val_loss"],
            fn=output_dir
            + "/loss_plot/"
            + "XL_oo_pp"
            + "_loss_over_epoch_line_plot_CV_group_"
            + str(i + 1)
            + "_rep_"
            + str(j + 1)
            + ".png",
        )
        loss_plot(
            train=history.history["custom_metric"],
            test=history.history["val_custom_metric"],
            fn=output_dir
            + "/loss_plot/"
            + "XL_oo_pp"
            + "_metric_over_epoch_line_plot_CV_group_"
            + str(i + 1)
            + "_rep_"
            + str(j + 1)
            + ".png",
        )
    best_model.save(
        output_dir
        + "/models/"
        + "XL_oo_pp"
        + "_best_model_CV_group_"
        + str(i + 1)
        + ".h5"
    )
    y_pred = best_model.predict(x_test, verbose=1).ravel()
    for k in range(len(y_test)):
        res.loc[len(res)] = list(
            map(
                str,
                [i + 1, y_df.loc[test_idx]["id"].to_numpy()[k], y_test[k], y_pred[k]],
            )
        )
stat.to_csv(
    output_dir + "/prediction/" + "XL_oo_pp" + "_train_test_CV_stats.csv", index=False
)
res.to_csv(
    output_dir + "/prediction/" + "XL_oo_pp" + "_train_test_CV_prediction_results.csv",
    index=False,
)

In [ ]:
res = pd.read_csv(
    output_dir + "/prediction/" + "XL_oo_pp" + "_train_test_CV_prediction_results.csv"
)
scatter_plot(
    res["y"],
    res["y_pred"],
    fn=output_dir + "/scatter_plot/" + "XL_oo_pp" + "_final_scatter_plot.png",
)

In [ ]:
res = pd.read_csv(
    output_dir + "/prediction/" + "XL_oo_pp" + "_train_test_CV_prediction_results.csv"
)
diff = np.array(res["y_pred"] - res["y"])
print(len(diff))
plt.hist(
    diff,
    bins=np.arange(round(min(diff)) - 1, round(max(diff)) + 1, 1),
    histtype="step",
    density=False,
)
plt.xlabel("predicted - measured")
plt.ylabel("data count")

In [ ]:
df = pd.DataFrame(None, columns=["a", "b", "c", "d", "e"])
df.loc[0] = [1, 2, 3, 4, 5]
df.loc[1] = [6, 7, 8, 9, 10]
df.loc[2] = [11, 12, 13, 14, 15]
df.loc[[0, 2], "c"]

---

# 2.Poly(A) length prediction

In [ ]:
output_dir = "/public/home/zhangyt/project/ai/polya_med"

In [ ]:
"mRNA", "coding_seq", "utr_seq", "polya_lengths", "polya_median"

In [ ]:
data_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_med/data/gene_3utr_polya.tsv", sep="\t"
)
mask = ~np.isnan(data_df["polya_median"])
data_df = data_df[mask]

In [ ]:
len(data_df)

In [ ]:
def encode_seq(seq: str) -> np.ndarray:
    """convert a DNA sequence to one-hot encoding matrix

    Args:
        seq (str): DNA sequence to encode

    Returns:
        np.ndarray: one-hot encoding matrix[A,C,G,T], N is [0,0,0,0]
    """
    mat = np.zeros((len(seq), 4))
    for i in range(len(seq)):
        if seq[i] == "A":
            mat[i, 0] = 1.0
        elif seq[i] == "C":
            mat[i, 1] = 1.0
        elif seq[i] == "G":
            mat[i, 2] = 1.0
        elif seq[i] == "T" or seq[i] == "U":
            mat[i, 3] = 1.0
    return mat

In [ ]:
x_encode_ls = []
id_ls = []
y_polya_ls = []
for i in range(len(data_df)):
    seq = data_df.loc[i, "utr_seq"].upper()
    if len(seq) <= 1053:
        norm_seq = "N" * (1053 - len(seq)) + seq + "AAA"
    else:
        norm_seq = seq[-1053:] + "AAA"
    id_ls.append(data_df.loc[i, "mRNA"])
    x_encode_ls.append(encode_seq(norm_seq))
    y_polya_ls.append(data_df.loc[i, "polya_median"])
x_ary = np.stack(x_encode_ls, axis=0)
y_df = pd.DataFrame({"id": id_ls, "y": y_polya_ls})

In [ ]:
def inspect(arr, name):
    arr = np.asarray(arr, dtype=np.float32)
    print(
        f"{name}: shape={arr.shape}",
        f"NaN? {np.isnan(arr).any()}",
        f"Inf? {np.isinf(arr).any()}",
        f"min={np.nanmin(arr):.3g}",
        f"max={np.nanmax(arr):.3g}",
    )


inspect(x_ary, "x_train")
inspect(y_df["y"], "y_train")

In [ ]:
nan_rows = y_df.index[y_df.isna().any(axis=1)]
print(nan_rows.tolist())

In [ ]:
mask = ~np.isnan(y_df["y"])
x_ary, y_df = x_ary[mask], y_df[mask]

In [ ]:
np.save("/public/home/zhangyt/project/ai/polya_med/data/x_input.npy", x_ary)
y_df.to_csv("/public/home/zhangyt/project/ai/polya_med/data/y_input.csv", index=False)

In [ ]:
x_ary = np.load("/public/home/zhangyt/project/ai/polya_med/data/x_input.npy")[20000:]
y_df = pd.read_csv("/public/home/zhangyt/project/ai/polya_med/data/y_input.csv")[20000:]

In [ ]:
print(x_ary.shape, y_df.shape)

In [ ]:
def custom_metric(y_true, y_pred):
    y_true_mean = keras.backend.mean(y_true, axis=0)
    y_pred_mean = keras.backend.mean(y_pred, axis=0)
    r_numerator = keras.backend.sum((y_true - y_true_mean) * (y_pred - y_pred_mean))
    x_square_sum = keras.backend.sum((y_true - y_true_mean) * (y_true - y_true_mean))
    y_square_sum = keras.backend.sum((y_pred - y_pred_mean) * (y_pred - y_pred_mean))
    r_denominator = keras.backend.sqrt(x_square_sum * y_square_sum)
    r = r_numerator / r_denominator
    metric_out = -(keras.backend.mean(r) ** 2)
    return metric_out

In [ ]:
def pal_model(
    filters: int = 64,
    kernel_size: int = 5,
    dropout_rate: float = 0.1,
    learning_rate: float = 0.001,
    l1: float = 0.01,
    l2: float = 0.000001,
) -> keras.Model:
    model = keras.Sequential(
        [
            layers.Input(shape=(1056, 4)),
            layers.Conv1D(filters=filters, kernel_size=kernel_size),
            layers.LayerNormalization(),
            layers.Activation(keras.activations.selu),
            layers.Conv1D(filters=filters, kernel_size=kernel_size),
            layers.Dropout(rate=dropout_rate),
            layers.MaxPooling1D(pool_size=2),
            layers.LayerNormalization(),
            layers.Activation(keras.activations.selu),
            layers.GRU(
                units=96,
                go_backwards=False,
                kernel_regularizer=keras.regularizers.l1_l2(l1, l2),
            ),
            layers.BatchNormalization(),
            layers.Activation(keras.activations.selu),
            layers.Dense(
                units=96,
                kernel_regularizer=keras.regularizers.l1_l2(l1, l2),
            ),
            layers.Dropout(rate=dropout_rate),
            layers.BatchNormalization(),
            layers.Activation(keras.activations.selu),
            layers.Dense(units=1),
        ]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae", custom_metric],
    )
    return model

In [ ]:
def train_model(x_train, y_train, x_test, y_test, session_id: str = "", **kwargs):
    model = pal_model(**kwargs)
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=output_dir + "/models/" + session_id + "_callback_best_model.h5",
            monitor="val_loss",
            mode="min",
            save_best_only=True,
        ),
        keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10),
    ]
    out = model.fit(
        x=x_train,
        y=y_train,
        batch_size=50,
        epochs=30,
        verbose=1,
        shuffle=True,
        callbacks=callbacks,
        validation_data=(x_test, y_test),
    )
    model = keras.models.load_model(
        output_dir + "/models/" + session_id + "_callback_best_model.h5",
        custom_objects={"custom_metric": custom_metric},
    )
    return out, model

In [ ]:
def loss_plot(train, test, fn):
    plt.clf()
    plt.plot(train, label="train")
    plt.plot(test, label="test")
    plt.legend()
    plt.xlabel("Epochs")
    plt.ylabel("Loss or metric")
    plt.savefig(fn)

In [ ]:
def scatter_plot(x, y, fn):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )
    plt.savefig(fn, dpi=300, bbox_inches="tight")

In [ ]:
stat = pd.DataFrame(
    None, columns=["group", "rep", "r2", "loss", "mae", "custom_metric"]
)
res = pd.DataFrame(None, columns=["group", "id", "y", "y_pred"])
kf = KFold(n_splits=10, shuffle=True, random_state=57)
for i, (train_idx, test_idx) in enumerate(kf.split(x_ary, y_df["y"])):
    best_r2 = 0
    for j in range(5):
        x_train = x_ary[train_idx]
        x_test = x_ary[test_idx]
        y_train = y_df.loc[train_idx, "y"].to_numpy()
        y_test = y_df.loc[test_idx, "y"].to_numpy()
        history, model = train_model(
            x_train, y_train, x_test, y_test, session_id="XL_oo_pp"
        )
        model.save(
            output_dir
            + "/models/"
            + "XL_oo_pp"
            + "_best_model_CV_group_"
            + str(i + 1)
            + "_rep_"
            + str(j + 1)
            + ".h5"
        )
        scores = model.evaluate(x_test, y_test, batch_size=50, verbose=1)
        y_pred = model.predict(x_test, verbose=1).ravel()
        r_value = stats.linregress(y_test, y_pred).rvalue
        if r_value**2 > best_r2:
            best_r2 = r_value**2
            best_model = model
        del model
        keras.backend.clear_session()
        stat.loc[len(stat)] = [i + 1, j + 1, round(r_value**2, 3)] + list(scores)
        loss_plot(
            train=history.history["loss"],
            test=history.history["val_loss"],
            fn=output_dir
            + "/loss_plot/"
            + "XL_oo_pp"
            + "_loss_over_epoch_line_plot_CV_group_"
            + str(i + 1)
            + "_rep_"
            + str(j + 1)
            + ".png",
        )
        loss_plot(
            train=history.history["custom_metric"],
            test=history.history["val_custom_metric"],
            fn=output_dir
            + "/loss_plot/"
            + "XL_oo_pp"
            + "_metric_over_epoch_line_plot_CV_group_"
            + str(i + 1)
            + "_rep_"
            + str(j + 1)
            + ".png",
        )
    best_model.save(
        output_dir
        + "/models/"
        + "XL_oo_pp"
        + "_best_model_CV_group_"
        + str(i + 1)
        + ".h5"
    )
    y_pred = best_model.predict(x_test, verbose=1).ravel()
    for k in range(len(y_test)):
        res.loc[len(res)] = list(
            map(
                str,
                [i + 1, y_df.loc[test_idx]["id"].to_numpy()[k], y_test[k], y_pred[k]],
            )
        )
stat.to_csv(
    output_dir + "/prediction/" + "XL_oo_pp" + "_train_test_CV_stats.csv", index=False
)
res.to_csv(
    output_dir + "/prediction/" + "XL_oo_pp" + "_train_test_CV_prediction_results.csv",
    index=False,
)

In [ ]:
res = pd.read_csv(
    "/public/home/zhangyt/project/ai/polya_med/prediction/median_train_test_CV_prediction_results.csv"
)
scatter_plot(
    res["y"],
    res["y_pred"],
    fn=output_dir + "/scatter_plot/" + "XL_oo_pp" + "_final_scatter_plot.png",
)

### test on extremum

In [ ]:
data_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_med/data/gene_3utr_polya.tsv", sep="\t"
)
mask = ~np.isnan(data_df["polya_median"])
data_df = data_df[mask].reset_index()

In [ ]:
"mRNA", "coding_seq", "utr_seq", "polya_lengths", "polya_median"

In [ ]:
mask = ~np.isnan(data_df["polya_median"])
data_df = data_df[mask]

In [ ]:
len(data_df)

In [ ]:
n = int(len(data_df) * 0.10)

bottom_idx = data_df.nsmallest(n, "polya_median").index
top_idx = data_df.nlargest(n, "polya_median").index

data_df["label"] = np.nan
data_df.loc[bottom_idx, "label"] = 0
data_df.loc[top_idx, "label"] = 1

In [ ]:
top_df = data_df[data_df["label"] == 1]
bottom_df = data_df[data_df["label"] == 0]

In [ ]:
top_df.head()

In [ ]:
extremum_df = pd.concat([top_df, bottom_df], ignore_index=True)

In [ ]:
len(extremum_df)

In [ ]:
def encode_seq(seq: str) -> np.ndarray:
    """convert a DNA sequence to one-hot encoding matrix

    Args:
        seq (str): DNA sequence to encode

    Returns:
        np.ndarray: one-hot encoding matrix[A,C,G,T], N is [0,0,0,0]
    """
    mat = np.zeros((len(seq), 4))
    for i in range(len(seq)):
        if seq[i] == "A":
            mat[i, 0] = 1.0
        elif seq[i] == "C":
            mat[i, 1] = 1.0
        elif seq[i] == "G":
            mat[i, 2] = 1.0
        elif seq[i] == "T" or seq[i] == "U":
            mat[i, 3] = 1.0
    return mat

In [ ]:
x_encode_ls = []
id_ls = []
y_polya_ls = []
y_label = []
for i in range(len(extremum_df)):
    cds = extremum_df.loc[i, "coding_seq"].upper()
    utr = extremum_df.loc[i, "utr_seq"].upper()
    seq = "".join([cds, utr])
    if len(seq) <= 1053:
        norm_seq = "N" * (1053 - len(seq)) + seq + "AAA"
    else:
        norm_seq = seq[-1053:] + "AAA"
    id_ls.append(extremum_df.loc[i, "mRNA"])
    x_encode_ls.append(encode_seq(norm_seq))
    y_polya_ls.append(extremum_df.loc[i, "polya_median"])
    y_label.append(extremum_df.loc[i, "label"])
x_ary = np.stack(x_encode_ls, axis=0)
y_df = pd.DataFrame({"id": id_ls, "y": y_polya_ls, "label": y_label})

In [ ]:
np.save("/public/home/zhangyt/project/ai/extremum_test/data/x_input.npy", x_ary)
y_df.to_csv(
    "/public/home/zhangyt/project/ai/extremum_test/data/y_input.csv", index=False
)

In [ ]:
y_df = pd.read_csv("/DataZhai/zhangyt/project/ai/extremum_test/data/y_input.csv")
y_pred = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/extremum_test/prediction/extremum_train_test_CV_prediction_results.csv"
)
merged_df = y_df.merge(y_pred, on=["id", "y"])
merged_df.head()

In [ ]:
merged_df["label_pred"] = merged_df["label_y"].map(round)
merged_df.head()

In [ ]:
tp = 0
fp = 0
tn = 0
fn = 0
for i in range(len(merged_df)):
    if (merged_df.loc[i, "label_x"] == 1) and (merged_df.loc[i, "label_pred"] == 1):
        tp += 1
    elif (merged_df.loc[i, "label_x"] == 0) and (merged_df.loc[i, "label_pred"] == 0):
        tn += 1
    elif (merged_df.loc[i, "label_x"] == 0) and (merged_df.loc[i, "label_pred"] == 1):
        fn += 1
    elif (merged_df.loc[i, "label_x"] == 1) and (merged_df.loc[i, "label_pred"] == 0):
        fp += 1
mat = np.array([[tp, fp], [fn, tn]])

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(mat, cmap="RdYlBu_r")
ax.set_xticks(range(2), labels=["long", "short"])
ax.set_yticks(range(2), labels=["long", "short"])
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, mat[i, j], ha="center", va="center", color="w")
plt.show()

In [ ]:
def iqr(ary):
    ary = np.array(ary)
    q3, q1 = np.percentile(ary, [75, 25])
    return q3 - q1

In [ ]:
iqr_ls = []
for i in range(len(data_df)):
    iqr_ls.append(iqr(eval(data_df.loc[i, "polya_lengths"])))

In [ ]:
len(iqr_ls)

In [ ]:
fig, ax = plt.subplots()
x = data_df["polya_median"]
y = iqr_ls
values = np.vstack([x, y])
kernel = stats.gaussian_kde(values)(values)
ax.scatter(x, y, marker=".", c=kernel, cmap="RdYlBu_r")
ax.set_xlabel("median")
ax.set_ylabel("IQR")
min_val = min(min(x), min(y))
max_val = max(max(x), max(y))
ax.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
plt.show()

# 3.poly(A) distribution

In [ ]:
data_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/gene_3utr_polya.tsv", sep="\t"
)
mask = ~np.isnan(data_df["polya_median"])
data_df = data_df[mask].reset_index()

In [ ]:
lengths = data_df["polya_lengths"].str.split(",").explode().astype(float)
len_min = lengths.min()
len_max = lengths.max()
print(len_min, len_max)

In [ ]:
len(lengths)

In [ ]:
sub = lengths
print(len(sub))
fig, ax = plt.subplots()
ax.hist(sub, bins=100, density=False, histtype="step")
ax.set_xlabel("polya_length")
ax.set_ylabel("count")

In [ ]:
import json

bins = np.arange(10, 330, 5)
bins = np.append(bins, np.inf)


def length_ratio(polya_str: str, bins=bins) -> str:
    lengths = np.fromstring(polya_str, sep=",")
    counts, _ = np.histogram(lengths, bins=bins)
    ratio = counts / counts.sum()
    return json.dumps(ratio.tolist())


data_df["polya_ratio"] = data_df["polya_lengths"].apply(length_ratio)

In [ ]:
data_df = data_df.drop("index", axis=1)

In [ ]:
data_df.to_csv(
    "/DataZhai/zhangyt/project/ai/datasets/polya_distribution.tsv",
    sep="\t",
    index=False,
)

## 1.Wasserstein-1D PCA

In [ ]:
dist_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/polya_distribution.tsv", sep="\t"
)

In [ ]:
count = dist_df["polya_lengths"].apply(lambda x: len(x.split(",")))
dist_df["count"] = count

In [ ]:
filtered_df = dist_df[dist_df["count"] >= 1000]
len(filtered_df)

In [ ]:
soybean_df = dist_df[dist_df["mRNA"].apply(lambda x: x.startswith("Glyma"))]

In [ ]:
len(soybean_df[soybean_df["count"] >= 200])

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from scipy.interpolate import interp1d

ratio_col = filtered_df["polya_ratio"].apply(eval)
X = np.vstack(ratio_col)

L, m = X.shape[1], 200
p_grid = (np.arange(m) + 0.5) / m
cum = X.cumsum(axis=1)


def to_quantile(cdf):
    f = interp1d(cdf, np.arange(L), bounds_error=False, fill_value=(0, L - 1))
    return f(p_grid)


Q = np.vstack([to_quantile(c) for c in cum])

pca = PCA(n_components=1)
t = pca.fit_transform(Q).ravel()
w = pca.components_[0]
Qbar = pca.mean_

# data_df["W1_PCA"] = t

In [ ]:
len(p_grid)

In [ ]:
data_df.to_csv(
    "/DataZhai/zhangyt/project/ai/datasets/polya_distribution.tsv",
    sep="\t",
    index=False,
)

In [ ]:
var_ratio = pca.explained_variance_ratio_[0]
print(var_ratio)

In [ ]:
from sklearn.manifold import MDS

md = MDS(dissimilarity="precomputed")
D = scipy.spatial.distance.squareform(
    scipy.spatial.distance.pdist(Q, metric="euclidean")
)
pos = md.fit_transform(D)
plt.scatter(pos[:, 0], pos[:, 1], c=t, cmap="coolwarm")

In [ ]:
def reconstruct_hist(t_i):
    Qrec = Qbar + t_i * w
    Fr = interp1d(Qrec, p_grid, bounds_error=False, fill_value=(0, 1))
    pdf = np.diff(np.hstack(([0], Fr(np.arange(L)))))
    return np.clip(pdf / pdf.sum(), 0, None)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.stairs(eval(dist_df.loc[2132, "polya_ratio"]), np.arange(10, 335, 5))
ax1.set(title="measured", xlabel="polya_length", ylabel="ratio", ylim=(0, 0.1))
ax2.stairs(reconstruct_hist(t[2132]), np.arange(10, 335, 5))
ax2.set(title="rebuilt", xlabel="polya_length", ylabel="ratio", ylim=(0, 0.1))

In [ ]:
fig, ax = plt.subplots()
ax.stairs(eval(dist_df.loc[132, "polya_ratio"]), np.arange(10, 335, 5))
ax.set(title="measured", xlabel="polya_length", ylabel="ratio", ylim=(0, 0.1))
ax.stairs(reconstruct_hist(t[132]), np.arange(10, 335, 5))

In [ ]:
from scipy.stats import wasserstein_distance
import scipy.spatial as ss
import scipy.stats as st


D_orig = ss.distance.squareform(
    ss.distance.pdist(X, metric=wasserstein_distance)
)  # Wasserstein Distance matrix
D_emb = ss.distance.squareform(
    ss.distance.pdist(t[:, None], metric="euclidean")
)  # Embedding Space Distance matrix
mask = np.triu_indices_from(D_orig, k=1)
spearman = st.spearmanr(D_orig[mask], D_emb[mask]).correlation
pearson = np.corrcoef(D_orig[mask], D_emb[mask])[0, 1]
stress = np.sqrt(
    ((D_orig - D_emb) ** 2).sum() / (D_orig**2).sum()
)  # Normalized Kruskal stress-1


def kl(p, q):  # mean KL Divergence
    return st.entropy(p, q)


kl_list = [kl(p, reconstruct_hist(ti)) for p, ti in zip(X, t)]
mean_kl = np.mean(kl_list)
var_ratio = pca.explained_variance_ratio_[0]  # Explained Var% PCA

In [ ]:
import scipy.spatial as ss
import scipy.stats as st
import random

N = 47992
pairs = random.sample(range(N * (N - 1) // 2), 100000)
mask = np.zeros(N * (N - 1) // 2, bool)
mask[pairs] = True
d_orig = ss.distance.pdist(X, wasserstein_distance)[mask]
d_emb = ss.distance.pdist(t[:, None], metric="euclidean")[mask]

spearman = st.spearmanr(d_orig, d_emb).correlation
var_ratio = pca.explained_variance_ratio_[0]
print(spearman)
print(var_ratio)

## 2.Wasserstein-AutoEncoder

### 1D

In [ ]:
import os, json, numpy as np, pandas as pd, tensorflow as tf
import keras

L = 64
LATENT = 1
BATCH = 256
EPOCHS = 30
BETA = 10.0


dist_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/polya_distribution.tsv", sep="\t"
)
count = dist_df["polya_lengths"].apply(lambda x: len(x.split(",")))
dist_df["count"] = count
filtered_df = dist_df[dist_df["count"] >= 100]
X = np.vstack(filtered_df["polya_ratio"].apply(json.loads)).astype("float32")

dataset = tf.data.Dataset.from_tensor_slices(X)
dataset = dataset.shuffle(len(X), seed=42).batch(BATCH).prefetch(tf.data.AUTOTUNE)


def wasserstein_1d(p, q):
    cdf_p = tf.cumsum(p, axis=-1)
    cdf_q = tf.cumsum(q, axis=-1)
    w1 = tf.reduce_sum(tf.abs(cdf_p - cdf_q), axis=-1)
    return tf.reduce_mean(w1)


def mmd_rbf(z, sigma=None):
    """Gaussian-kernel MMD^2, z: [B,1]
    prior ~ N(0,1). sigma 自动估计。"""
    if sigma is None:
        sigma = tf.maximum(tf.math.reduce_mean(tf.math.square(z)), 1e-3)

    def kernel(a, b):
        return tf.exp(-tf.math.square(a - b) / (2.0 * sigma))

    prior = tf.random.normal(tf.shape(z))
    xx = kernel(tf.expand_dims(z, 0), tf.expand_dims(z, 1))
    pp = kernel(tf.expand_dims(prior, 0), tf.expand_dims(prior, 1))
    xp = kernel(tf.expand_dims(z, 0), tf.expand_dims(prior, 1))
    return tf.reduce_mean(xx) + tf.reduce_mean(pp) - 2.0 * tf.reduce_mean(xp)


class Encoder(keras.Model):
    def __init__(self):
        super().__init__()
        self.net = keras.Sequential(
            [
                keras.layers.Dense(128, activation="gelu"),
                keras.layers.Dense(64, activation="gelu"),
            ]
        )
        self.mu = keras.layers.Dense(LATENT)
        self.logv = keras.layers.Dense(LATENT)

    def call(self, x):
        h = self.net(x)
        return self.mu(h), self.logv(h)


class Decoder(keras.Model):
    def __init__(self):
        super().__init__()
        self.net = keras.Sequential(
            [
                keras.layers.Dense(64, activation="gelu"),
                keras.layers.Dense(128, activation="gelu"),
                keras.layers.Dense(L),
            ]
        )

    def call(self, z):
        logits = self.net(z)
        return tf.nn.softmax(logits, axis=-1)


class WAE1D(keras.Model):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.dec = Decoder()
        self.metric_kl = keras.metrics.Mean(name="mean_KL")

    def compile(self, optimizer, **kwargs):
        super().compile(**kwargs)
        self.opt = optimizer

    def train_step(self, x):
        with tf.GradientTape() as tape:
            mu, logv = self.enc(x)
            eps = tf.random.normal(tf.shape(mu))
            z = mu + tf.exp(0.5 * logv) * eps
            xhat = self.dec(z)
            loss_rec = wasserstein_1d(x, xhat)
            loss_mmd = mmd_rbf(z)
            loss = loss_rec + BETA * loss_mmd  # loss = Wasserstein‑1 + β·MMD

            # 记录 KL(orig||recon)
            kl = tf.reduce_mean(
                tf.reduce_sum(
                    x * (tf.math.log(x + 1e-8) - tf.math.log(xhat + 1e-8)), axis=-1
                )
            )

        grads = tape.gradient(loss, self.trainable_variables)
        self.opt.apply_gradients(zip(grads, self.trainable_variables))
        self.metric_kl.update_state(kl)
        return {
            "loss": loss,
            "rec": loss_rec,
            "mmd": loss_mmd,
            "kl": self.metric_kl.result(),
        }

    def test_step(self, x):
        mu, logv = self.enc(x)
        eps = tf.random.normal(tf.shape(mu))
        z = mu + tf.exp(0.5 * logv) * eps
        xhat = self.dec(z)
        loss_rec = wasserstein_1d(x, xhat)
        loss_mmd = mmd_rbf(z)
        loss = loss_rec + BETA * loss_mmd
        kl = tf.reduce_mean(
            tf.reduce_sum(
                x * (tf.math.log(x + 1e-8) - tf.math.log(xhat + 1e-8)), axis=-1
            )
        )
        return {"loss": loss, "rec": loss_rec, "mmd": loss_mmd, "kl": kl}

    def encode_mean(self, x):
        mu, _ = self.enc(x)
        return mu

    def reconstruct(self, z):
        z = tf.convert_to_tensor(z, dtype=tf.float32)
        if len(z.shape) == 0:
            z = tf.reshape(z, (1, 1))
        return self.dec(z).numpy()


model = WAE1D()
model.compile(optimizer=keras.optimizers.Adam(3e-3))
callback = keras.callbacks.EarlyStopping("loss", patience=5, restore_best_weights=True)
model.fit(dataset, epochs=EPOCHS, callbacks=[callback])


emb = model.encode_mean(X).numpy().ravel()
filtered_df.loc[:, "tail_embed_WAE"] = emb

# rebuild
test_recon = model.reconstruct(emb[0])[0]
print("原 hist 前10:", X[0][:10])
print("重建前10:", test_recon[:10])

filtered_df.to_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/wae/wae1d_embed.tsv",
    sep="\t",
    index=False,
)

In [ ]:
filtered_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/wae/wae1d_embed.tsv", sep="\t"
)
model

In [ ]:
fig, ax = plt.subplots()
ax.stairs(eval(filtered_df.loc[132, "polya_ratio"]), np.arange(10, 335, 5))
ax.set(xlabel="polya_length", ylabel="ratio", ylim=(0, 0.1))
ax.stairs(model.reconstruct(emb[132])[0], np.arange(10, 335, 5))

In [ ]:
fig, ax = plt.subplots()
ax.stairs(eval(filtered_df.loc[1232, "polya_ratio"]), np.arange(10, 335, 5))
ax.set(xlabel="polya_length", ylabel="ratio", ylim=(0, 0.1))
ax.stairs(model.reconstruct(emb[1232])[0], np.arange(10, 335, 5))

In [ ]:
fig, ax = plt.subplots()
ax.stairs(eval(filtered_df.loc[732, "polya_ratio"]), np.arange(10, 335, 5))
ax.set(xlabel="polya_length", ylabel="ratio", ylim=(0, 0.1))
ax.stairs(model.reconstruct(emb[732])[0], np.arange(10, 335, 5))

### 2D

In [ ]:
soybean_df = dist_df[dist_df["mRNA"].apply(lambda x: x.startswith("Glyma"))]

In [ ]:
import json, numpy as np, pandas as pd, tensorflow as tf
import keras

L = 64
LATENT = 2
BATCH = 50
EPOCHS = 200
BETA = 0.8


filtered_df = dist_df[dist_df["count"] >= 200]
X = np.vstack(filtered_df["polya_ratio"].apply(json.loads)).astype("float32")
X += 1e-4
X /= X.sum(axis=1, keepdims=True)

dataset = (
    tf.data.Dataset.from_tensor_slices(X)
    .shuffle(len(X), seed=42)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)


def wasserstein_1d(p, q):
    cdf_p = tf.cumsum(p, axis=-1)
    cdf_q = tf.cumsum(q, axis=-1)
    w1 = tf.reduce_sum(tf.abs(cdf_p - cdf_q), axis=-1)
    return tf.reduce_mean(w1)


def mmd_rbf(z, sigma=None):
    if sigma is None:
        sigma = tf.maximum(tf.math.reduce_mean(tf.square(z)), 1e-3)

    def k(a, b):
        return tf.exp(-tf.square(a - b) / (2.0 * sigma))

    prior = tf.random.normal(tf.shape(z))
    xx = k(z[:, None, :], z[None, :, :])
    pp = k(prior[:, None, :], prior[None, :, :])
    xp = k(z[:, None, :], prior[None, :, :])
    return tf.reduce_mean(xx) + tf.reduce_mean(pp) - 2.0 * tf.reduce_mean(xp)


@keras.saving.register_keras_serializable()
class Encoder(keras.Model):

    def __init__(self, LATENT=2):
        super().__init__()
        self.backbone = keras.Sequential(
            [
                keras.layers.Input(shape=(L,)),
                keras.layers.Dense(256, activation="gelu"),
                keras.layers.Dense(128, activation="gelu"),
                keras.layers.Dense(64, activation="gelu"),
            ]
        )
        self.mu = keras.layers.Dense(LATENT)
        self.logv = keras.layers.Dense(LATENT)

    def call(self, x):
        h = self.backbone(x)
        return self.mu(h), self.logv(h)


@keras.saving.register_keras_serializable()
class Decoder(keras.Model):

    def __init__(self, tau=0.6):
        super().__init__()
        self.tau = tau
        self.net = keras.Sequential(
            [
                keras.layers.Input(shape=(LATENT,)),
                keras.layers.Dense(256, activation="gelu"),
                keras.layers.Dense(256, activation="gelu"),
                keras.layers.Dense(128, activation="gelu"),
                keras.layers.Dense(L),
            ]
        )

    def call(self, z):
        logits = self.net(z) / self.tau
        return tf.nn.softmax(logits, axis=-1)


@keras.saving.register_keras_serializable()
class WAE2D(keras.Model):

    def __init__(self, LATENT=2, tau=0.6, **kwargs):
        super().__init__(**kwargs)
        self.LATENT = LATENT
        self.tau = tau
        self.enc = Encoder(LATENT)
        self.dec = Decoder(tau)
        self.kl_metric = keras.metrics.Mean(name="mean_KL")

    def compile(self, optimizer):
        super().compile()
        self.optimizer = optimizer

    def train_step(self, x):
        with tf.GradientTape() as tape:
            mu, logv = self.enc(x)
            eps = tf.random.normal(tf.shape(mu))
            z = mu + tf.exp(0.5 * logv) * eps
            x_hat = self.dec(z)

            loss_rec = wasserstein_1d(x, x_hat)
            loss_mmd = mmd_rbf(z)
            loss = loss_rec + BETA * loss_mmd

            kl = tf.reduce_mean(
                tf.reduce_sum(
                    x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8)), axis=-1
                )
            )

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.kl_metric.update_state(kl)
        return {
            "loss": loss,
            "rec": loss_rec,
            "mmd": loss_mmd,
            "kl": self.kl_metric.result(),
        }

    def encode_mean(self, x):
        mu, _ = self.enc(x)
        return mu

    def reconstruct(self, z):
        z = tf.convert_to_tensor(z, dtype=tf.float32)
        if z.ndim == 1:
            z = z[None, :]
        return self.dec(z).numpy()

    def call(self, inputs, training=False):
        mu, _ = self.enc(inputs, training=training)
        return self.dec(mu, training=training)

    def get_config(self):
        return {
            "LATENT": self.LATENT,
            "tau": self.tau,
            **super().get_config(),
        }

    def get_build_config(self):
        return {"input_shape": (None, 64)}

    @classmethod
    def build_from_config(cls, config):
        model = cls(**config)
        dummy = tf.zeros((1, 64), dtype="float32")
        model(dummy)
        return model


model = WAE2D()
model(tf.zeros((1, L)))
optimizer = keras.optimizers.Adam(1e-3)
model.compile(optimizer=optimizer)
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="/DataZhai/zhangyt/project/ai/polya_dist/multi/wae2d",
        monitor="loss",
        mode="min",
        save_best_only=True,
        save_format="tf",
    ),
    keras.callbacks.EarlyStopping("loss", patience=15, restore_best_weights=True),
]
model.fit(dataset, verbose=1, epochs=EPOCHS, callbacks=callbacks)

z_mean = model.encode_mean(X).numpy()
filtered_df[["embed_z1", "embed_z2"]] = z_mean
filtered_df.to_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/wae2d_embed.tsv",
    sep="\t",
    index=False,
)
keras.backend.clear_session()

In [ ]:
embed_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/wae2d_embed.tsv", sep="\t"
)

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr

X = np.vstack(embed_df["polya_ratio"].apply(json.loads)).astype("float32")
Z = embed_df[["embed_z1", "embed_z2"]].values.astype("float32")

r2 = []
for l in range(L):
    y = X[:, l]
    lr = LinearRegression().fit(Z, y)
    r2.append(lr.score(Z, y))
z1 = [pearsonr(Z[:, 0], X[:, l])[0] for l in range(L)]
z2 = [pearsonr(Z[:, 1], X[:, l])[0] for l in range(L)]

In [ ]:
print("平均 R² :", np.mean(r2))
print("z1 相关系数前 10 bin:", np.round(z1[:10], 3))

In [ ]:
from scipy.stats import pearsonr

corr = np.zeros((2, L), dtype="float32")

for l in range(L):
    corr[0, l] = pearsonr(Z[:, 0], X[:, l])[0]
    corr[1, l] = pearsonr(Z[:, 1], X[:, l])[0]

bin_edges = np.arange(10, 330, 5)
tick_pos = np.arange(0, 64, 8)
tick_labels = bin_edges[tick_pos]

plt.figure(figsize=(12, 2.5))
im = plt.imshow(
    corr, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1, interpolation="nearest"
)

plt.yticks([0, 1], ["z1", "z2"], fontsize=12)
plt.xticks(tick_pos, tick_labels, fontsize=10)
plt.xlabel("poly(A) length (nt)", fontsize=12)
plt.colorbar(im, label="Pearson r", shrink=0.8)
plt.title("Correlation heat-map: latent coordinates vs tail-length bins", pad=10)
plt.tight_layout()

In [ ]:
if "polya_median" in embed_df.columns:
    color = embed_df["polya_median"].values.astype("float32")
    c_label = "poly(A) median (nt)"
else:
    # compute expectation = Σ bin_center * ratio
    bin_centers = np.arange(10, 10 + 5 * 64, 5)
    ratios = np.vstack(embed_df["polya_ratio"].apply(json.loads)).astype("float32")
    color = (ratios * bin_centers).sum(axis=1)
    c_label = "median poly(A) length (nt)"

# -- scatter plot --
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=color, cmap="viridis", s=20, alpha=0.8)
plt.colorbar(sc, label=c_label)
plt.xlabel("latent z1")
plt.ylabel("latent z2")
plt.title("2-D latent map of poly(A) tail profiles")
plt.tight_layout()

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import json
import random
import ot

assert {"polya_ratio", "embed_z1", "embed_z2"}.issubset(
    embed_df.columns
), "embed_df must contain polya_ratio, embed_z1, embed_z2"

# 1) latent coords & histograms
Z = embed_df[["embed_z1", "embed_z2"]].values.astype("float32")
H = np.vstack(embed_df["polya_ratio"].apply(json.loads)).astype("float32")
bin_centers = np.arange(12.5, 12.5 + 5 * 64, 5, dtype="float32")

# 2) random pair sampling
N = len(embed_df)
M = min(6000, N * (N - 1) // 4)
pairs = random.sample([(i, j) for i in range(N) for j in range(i + 1, N)], M)

latent_d = np.empty(M, dtype="float32")
w2_d = np.empty(M, dtype="float32")

for k, (i, j) in enumerate(pairs):
    latent_d[k] = np.linalg.norm(Z[i] - Z[j])
    w2_d[k] = np.sqrt(ot.wasserstein_1d(bin_centers, bin_centers, H[i], H[j], p=2))

# 3) linear regression
x = latent_d.reshape(-1, 1)
y = w2_d
lr = LinearRegression().fit(x, y)
coef, intercept = lr.coef_[0], lr.intercept_
xs = np.linspace(0, x.max(), 200, dtype="float32")[:, None]
ys = lr.predict(xs)
r, _ = pearsonr(latent_d, w2_d)
r2 = r**2

# 4) plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, s=10, c="#f6b26b", alpha=0.35, marker="x", label="pairs")
plt.plot(
    xs,
    ys,
    color="red",
    lw=2.5,
    label=f"fit  y = {coef:.3f}·x + {intercept:.3f}\nPearson r² = {r2:.3f}",
)
plt.xlabel(r"latent $|\Delta s|$", fontsize=12)
plt.ylabel(r"$W_2$", fontsize=12)
plt.title("Distance linearization", fontsize=14, pad=12)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
len(embed_df)

In [ ]:
soybean_model = keras.models.load_model(
    "/DataZhai/zhangyt/project/ai/polya_dist/soybean/soybean_wae2d.keras",
    custom_objects={"WAE2D": WAE2D},
    compile=False,
)

In [ ]:
L = 64
LATENT = 2
BATCH = 50
EPOCHS = 200
BETA = 0.8


def wasserstein_1d(p, q):
    cdf_p = tf.cumsum(p, axis=-1)
    cdf_q = tf.cumsum(q, axis=-1)
    w1 = tf.reduce_sum(tf.abs(cdf_p - cdf_q), axis=-1)
    return tf.reduce_mean(w1)


def mmd_rbf(z, sigma=None):
    if sigma is None:
        sigma = tf.maximum(tf.math.reduce_mean(tf.square(z)), 1e-3)

    def k(a, b):
        return tf.exp(-tf.square(a - b) / (2.0 * sigma))

    prior = tf.random.normal(tf.shape(z))
    xx = k(z[:, None, :], z[None, :, :])
    pp = k(prior[:, None, :], prior[None, :, :])
    xp = k(z[:, None, :], prior[None, :, :])
    return tf.reduce_mean(xx) + tf.reduce_mean(pp) - 2.0 * tf.reduce_mean(xp)


@keras.saving.register_keras_serializable()
class Encoder(keras.Model):

    def __init__(self, LATENT=2):
        super().__init__()
        self.backbone = keras.Sequential(
            [
                keras.layers.Input(shape=(L,)),
                keras.layers.Dense(256, activation="gelu"),
                keras.layers.Dense(128, activation="gelu"),
                keras.layers.Dense(64, activation="gelu"),
            ]
        )
        self.mu = keras.layers.Dense(LATENT)
        self.logv = keras.layers.Dense(LATENT)

    def call(self, x):
        h = self.backbone(x)
        return self.mu(h), self.logv(h)


@keras.saving.register_keras_serializable()
class Decoder(keras.Model):

    def __init__(self, tau=0.6):
        super().__init__()
        self.tau = tau
        self.net = keras.Sequential(
            [
                keras.layers.Input(shape=(LATENT,)),
                keras.layers.Dense(256, activation="gelu"),
                keras.layers.Dense(256, activation="gelu"),
                keras.layers.Dense(128, activation="gelu"),
                keras.layers.Dense(L),
            ]
        )

    def call(self, z):
        logits = self.net(z) / self.tau
        return tf.nn.softmax(logits, axis=-1)


@keras.saving.register_keras_serializable()
class WAE2D(keras.Model):

    def __init__(self, LATENT=2, tau=0.6, **kwargs):
        super().__init__(**kwargs)
        self.LATENT = LATENT
        self.tau = tau
        self.enc = Encoder(LATENT)
        self.dec = Decoder(tau)
        self.kl_metric = keras.metrics.Mean(name="mean_KL")

    def compile(self, optimizer):
        super().compile()
        self.optimizer = optimizer

    def train_step(self, x):
        with tf.GradientTape() as tape:
            mu, logv = self.enc(x)
            eps = tf.random.normal(tf.shape(mu))
            z = mu + tf.exp(0.5 * logv) * eps
            x_hat = self.dec(z)

            loss_rec = wasserstein_1d(x, x_hat)
            loss_mmd = mmd_rbf(z)
            loss = loss_rec + BETA * loss_mmd

            kl = tf.reduce_mean(
                tf.reduce_sum(
                    x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8)), axis=-1
                )
            )

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.kl_metric.update_state(kl)
        return {
            "loss": loss,
            "rec": loss_rec,
            "mmd": loss_mmd,
            "kl": self.kl_metric.result(),
        }

    def encode_mean(self, x):
        mu, _ = self.enc(x)
        return mu

    def reconstruct(self, z):
        z = tf.convert_to_tensor(z, dtype=tf.float32)
        if z.ndim == 1:
            z = z[None, :]
        return self.dec(z).numpy()

    def call(self, inputs, training=False):
        mu, _ = self.enc(inputs, training=training)
        return self.dec(mu, training=training)

    def get_config(self):
        return {
            "LATENT": self.LATENT,
            "tau": self.tau,
            **super().get_config(),
        }

    def get_build_config(self):
        return {"input_shape": (None, 64)}

    @classmethod
    def build_from_config(cls, config):
        model = cls(**config)
        dummy = tf.zeros((1, 64), dtype="float32")
        model(dummy)
        return model


model = keras.models.load_model(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/wae2d",
    compile=False,
)
# 看第一层权重的均值
print("first weight mean:", np.mean(model.weights[0].numpy()))

In [ ]:
wae2d_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/wae_kfold_embeddings.tsv", sep="\t"
)

In [ ]:
x = np.asarray(eval(wae2d_df.loc[5564, "hist_true"]), dtype="float32")
x_hat = np.asarray(eval(wae2d_df.loc[5564, "hist_pred"]), dtype="float32")
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="rebuilt")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio", ylim=(0, 0.10))
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
x = np.asarray(eval(embed_df.loc[4070, "polya_ratio"]), dtype="float32")
x_hat = model.reconstruct(z_mean.loc[4070].to_numpy())[0].astype("float32")
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="rebuilt")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio", ylim=(0, 0.10))
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance

p = np.array(
    [
        0.04040403291583061,
        0.0471380352973938,
        0.04040403291583061,
        0.020202020183205605,
        0.043771032243967056,
        0.04040403291583061,
        0.030303025618195534,
        0.030303025618195534,
        0.04040403291583061,
        0.02693602256476879,
        0.02693602256476879,
        0.01683501899242401,
        0.01683501899242401,
        0.020202020183205605,
        0.013468016870319843,
        0.020202020183205605,
        0.013468016870319843,
        0.02693602256476879,
        0.020202020183205605,
        0.020202020183205605,
        0.01010101567953825,
        0.01010101567953825,
        0.020202020183205605,
        0.01683501899242401,
        0.030303025618195534,
        0.023569021373987198,
        0.020202020183205605,
        0.006734013091772795,
        0.020202020183205605,
        0.023569021373987198,
        0.006734013091772795,
        0.0033670116681605577,
        0.0033670116681605577,
        0.0033670116681605577,
        0.01683501899242401,
        0.013468016870319843,
        0.020202020183205605,
        0.0033670116681605577,
        0.01683501899242401,
        0.0033670116681605577,
        0.006734013091772795,
        0.013468016870319843,
        0.013468016870319843,
        0.01683501899242401,
        0.01010101567953825,
        0.013468016870319843,
        0.0033670116681605577,
        0.0033670116681605577,
        0.0033670116681605577,
        0.0033670116681605577,
        0.01683501899242401,
        0.01010101567953825,
        0.0033670116681605577,
        9.999995498333192e-09,
        0.006734013091772795,
        0.006734013091772795,
        0.01010101567953825,
        9.999995498333192e-09,
        0.0033670116681605577,
        9.999995498333192e-09,
        9.999995498333192e-09,
        9.999995498333192e-09,
        0.006734013091772795,
        0.020202020183205605,
    ]
)

q = np.array(
    [
        0.033191852271556854,
        0.05103962495923042,
        0.047990016639232635,
        0.041323471814394,
        0.03984157741069794,
        0.04083874821662903,
        0.03945406898856163,
        0.03845580294728279,
        0.034923579543828964,
        0.03307703137397766,
        0.03295315057039261,
        0.0323001965880394,
        0.031335968524217606,
        0.030173348262906075,
        0.029619738459587097,
        0.02916717156767845,
        0.02812288887798786,
        0.028896357864141464,
        0.02731262892484665,
        0.026992078870534897,
        0.025729067623615265,
        0.025560997426509857,
        0.023356277495622635,
        0.022910216823220253,
        0.021233361214399338,
        0.020073287189006805,
        0.01896752417087555,
        0.017730288207530975,
        0.015907960012555122,
        0.014340247958898544,
        0.013263787142932415,
        0.011797870509326458,
        0.0110426414757967,
        0.009239386767148972,
        0.007015496026724577,
        0.007283276412636042,
        0.005678762216120958,
        0.004937503486871719,
        0.004047406371682882,
        0.0035533313639461994,
        0.0024100381415337324,
        0.002472224645316601,
        0.0018991118995472789,
        0.0015963023761287332,
        0.0014262693002820015,
        0.001139176543802023,
        0.0010597796645015478,
        0.0008469783351756632,
        0.0008181399316526949,
        0.0005768954870291054,
        0.0006469341460615396,
        0.0005363301024772227,
        0.00045266191591508687,
        0.0004088705172762275,
        0.000500031397677958,
        0.0002535326930228621,
        0.0003349899488966912,
        0.00033247924875468016,
        0.00018231221474707127,
        0.00027949141804128885,
        0.00016455166041851044,
        0.00022954658197704703,
        0.0002348557027289644,
        0.0005204440676607192,
    ]
)

# normalise (should already sum to 1)
p /= p.sum()
q /= q.sum()

# 1D supports (bin centres)
x = np.arange(len(p))

# W1 & W2
w1 = wasserstein_distance(x, x, p, q)
w2 = np.sqrt(np.sum(p * np.cumsum(p) * 0))  # placeholder


# closed-form 1D W2: sqrt(sum_i p_i * (CDF_p[i]-CDF_q[i])^2 / sum p) but easier: map to sorted list of samples proportional to probs
def w2_discrete(p, q, x):
    # generate samples to approximate; use cdf matching in prob mass quantile
    cdf_p = np.cumsum(p)
    cdf_q = np.cumsum(q)
    # ensure last=1
    N = 10000
    u = (np.arange(N) + 0.5) / N
    xp = np.interp(u, cdf_p, x)
    xq = np.interp(u, cdf_q, x)
    return np.sqrt(np.mean((xp - xq) ** 2))


w2 = w2_discrete(p, q, x)

# plot histograms
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].bar(x - 0.2, p, width=0.4, label="p")
ax[0].bar(x + 0.2, q, width=0.4, label="q")
ax[0].set_title("Distributions")
ax[0].legend()

# plot CDFs and area
cdf_p = np.cumsum(p)
cdf_q = np.cumsum(q)
ax[1].step(x, cdf_p, where="post", label="CDF p")
ax[1].step(x, cdf_q, where="post", label="CDF q")
ax[1].fill_between(x, cdf_p, cdf_q, alpha=0.3)
ax[1].set_title(f"W1 ≈ {w1:.3f},  W2 ≈ {w2:.3f}")
ax[1].legend()

plt.tight_layout()
plt.show()

## Res-GRU-MHA

In [ ]:
output_dir = "/public/home/zhangyt/project/ai/polya_med"

In [ ]:
"mRNA", "coding_seq", "utr_seq", "polya_lengths", "polya_median"

In [ ]:
data_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/gene_3utr_polya.tsv", sep="\t"
)

In [ ]:
length = []
for i in range(len(data_df)):
    length.append(len(data_df.iloc[i, 1]) + len(data_df.iloc[i, 2]))
data_df["length"] = length

In [ ]:
data_df["count"] = data_df["polya_lengths"].apply(lambda x: len(str(x).split(",")))

In [ ]:
len(data_df[(data_df["count"] >= 200) & (data_df["length"] >= 2500)])

In [ ]:
data_df[(data_df["count"] >= 200) & (data_df["length"] >= 2500)].head()

In [ ]:
embed_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/wae2d_embed.tsv", sep="\t"
)

In [ ]:
merged_df = data_df.merge(embed_df, on=["mRNA", "polya_lengths", "count"])

In [ ]:
merged_df.head()

In [ ]:
merged_df.to_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/data/embed.tsv", sep="\t", index=False
)

mRNA	coding_seq	utr_seq	polya_lengths	polya_median	length	count	polya_ratio	embed_z1	embed_z2

In [ ]:
def encode_seq(seq: str) -> np.ndarray:
    """convert a DNA sequence to one-hot encoding matrix

    Args:
        seq (str): DNA sequence to encode

    Returns:
        np.ndarray: one-hot encoding matrix[A,C,G,T], N is [0,0,0,0]
    """
    mat = np.zeros((len(seq), 4))
    for i in range(len(seq)):
        if seq[i] == "A":
            mat[i, 0] = 1.0
        elif seq[i] == "C":
            mat[i, 1] = 1.0
        elif seq[i] == "G":
            mat[i, 2] = 1.0
        elif seq[i] == "T" or seq[i] == "U":
            mat[i, 3] = 1.0
    return mat

In [ ]:
filtered_df = merged_df[
    (merged_df["count"] >= 200) & (merged_df["length"] >= 2500)
].reset_index()
y_df = filtered_df[["mRNA", "polya_ratio", "embed_z1", "embed_z2"]]
x_encode_ls = []
for i in range(len(filtered_df)):
    seq = "".join(
        [filtered_df.loc[i, "coding_seq"], filtered_df.loc[i, "utr_seq"]]
    ).upper()
    if len(seq) < 3000:
        norm_seq = "".join(["N" * (3000 - len(seq)), seq])
    else:
        norm_seq = seq[-3000:]
    x_encode_ls.append(encode_seq(norm_seq))
x_ary = np.stack(x_encode_ls, axis=0)

In [ ]:
np.save("/DataZhai/zhangyt/project/ai/polya_dist/data/x_input.npy", x_ary)
y_df.to_csv("/DataZhai/zhangyt/project/ai/polya_dist/data/y_input.csv", index=False)

In [ ]:
x_ary = np.load("/DataZhai/zhangyt/project/ai/polya_dist/data/x_input.npy")
y_df = pd.read_csv("/DataZhai/zhangyt/project/ai/polya_dist/data/y_input.csv")

In [ ]:
print(x_ary.shape, y_df.shape)

In [ ]:
L_NT = 3000  # 原始序列长度 (nt)
D_RES = 128  # 卷积通道 / GRU 隐单元
D_MODEL = 256  # Transformer 维度
N_HEAD = 8  # Multi-Head 数
N_LATENT = 2  # 输出 z 维度
DILATIONS = [1, 2, 4]


# ---------- Residual Dilated Conv block ----------
def res_block(x, k_size, dilation, name):
    h = keras.layers.Conv1D(
        D_RES,
        k_size,
        padding="same",
        dilation_rate=dilation,
        activation="gelu",
        name=f"{name}_conv1",
    )(x)
    h = keras.layers.BatchNormalization(name=f"{name}_bn1")(h)
    h = keras.layers.Conv1D(
        D_RES,
        k_size,
        padding="same",
        dilation_rate=dilation,
        activation=None,
        name=f"{name}_conv2",
    )(h)
    h = keras.layers.BatchNormalization(name=f"{name}_bn2")(h)
    return keras.layers.Activation("gelu", name=f"{name}_out")(x + h)


# ---------- Transformer block ----------
def transformer_block(x, name):
    # LayerNorm + MH-Attention
    h1 = keras.layers.LayerNormalization(epsilon=1e-5, name=f"{name}_ln1")(x)
    h1 = keras.layers.MultiHeadAttention(
        num_heads=N_HEAD, key_dim=D_MODEL // N_HEAD, dropout=0.1, name=f"{name}_mha"
    )(h1, h1)
    x = x + h1  # residual

    # FFN
    h2 = keras.layers.LayerNormalization(epsilon=1e-5, name=f"{name}_ln2")(x)
    h2 = keras.layers.Dense(4 * D_MODEL, activation="gelu", name=f"{name}_ffn1")(h2)
    h2 = keras.layers.Dense(D_MODEL, name=f"{name}_ffn2")(h2)
    return x + h2  # residual


# ------------- Model definition ------------------
inp = keras.Input(shape=(L_NT, 4), dtype="float32", name="input")

# 0) ConvStem -- stride 2 ×2   (3000→750)
x = keras.layers.Conv1D(
    D_RES, 11, strides=2, padding="same", activation="gelu", name="stem_conv1"
)(inp)
x = keras.layers.Conv1D(
    D_RES, 11, strides=2, padding="same", activation="gelu", name="stem_conv2"
)(x)

# 1) Residual Dilated CNN      (750→375 通过后续池化，下采样=2)
for i, d in enumerate(DILATIONS):
    x = res_block(x, k_size=11 if d < 4 else 15, dilation=d, name=f"res{i+1}")

# 2) Bi-GRU ×2
x = keras.layers.Bidirectional(
    keras.layers.GRU(D_RES, return_sequences=True), name="bigru1"
)(x)
x = keras.layers.Bidirectional(
    keras.layers.GRU(D_RES, return_sequences=True), name="bigru2"
)(x)

# 3) Avg-Pool /3  (375→125)
x = keras.layers.AveragePooling1D(pool_size=3, name="avgpool")(x)

# 4) 2 × Transformer
x = transformer_block(x, name="trans1")
x = transformer_block(x, name="trans2")

# 5) Global pool + head
g = keras.layers.GlobalAveragePooling1D(name="gap")(x)
g = keras.layers.Dense(128, activation="gelu", name="dense128")(g)
out = keras.layers.Dense(N_LATENT, name="embed")(g)

model = keras.Model(inp, out, name="ResGRUTransformer")
model.summary()

# -------- compile (示范，用你的自定义损失替换) ----------
model.compile(
    optimizer=keras.optimizers.AdamW(3e-4),
    loss="mse",  # 训练时可换为 λ*MSE(z)+λ2*W1(hist)
    metrics=[keras.metrics.RootMeanSquaredError()],
)

### rgt-wae

In [ ]:
res = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/train_val_CV_result.tsv", sep="\t"
)

In [ ]:
print(res.columns)

In [ ]:
x = np.asarray(eval(res.loc[4070, "polya_ratio"]), dtype="float32")
x_hat = np.asarray(eval(res.loc[4070, "reconstruct"]), dtype="float32")
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="rebuilt")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio", ylim=(0, 0.10))
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
x = np.asarray(eval(res.loc[13, "polya_ratio"]), dtype="float32")
x_hat = np.asarray(eval(res.loc[13, "reconstruct"]), dtype="float32")
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="rebuilt")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio", ylim=(0, 0.10))
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
def scatter_plot(x, y):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )
    # plt.savefig(fn, dpi=300, bbox_inches="tight")

In [ ]:
z1 = res["z"].apply(lambda x: eval(x)[0])
z1_pred = res["z_pred"].apply(lambda x: eval(x)[0])
scatter_plot(z1, z1_pred)

In [ ]:
z2 = res["z"].apply(lambda x: eval(x)[1])
z2_pred = res["z_pred"].apply(lambda x: eval(x)[1])
scatter_plot(z2, z2_pred)

### rgt-direct

In [ ]:
res = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/multi/direct64_train_val_CV_result.tsv",
    sep="\t",
)
med = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/gene_3utr_polya.species.tsv",
    sep="\t",
)

In [ ]:
print(res.columns)

In [ ]:
res_med = res.merge(med[["mRNA", "polya_median"]], on=["mRNA"])

In [ ]:
x = np.asarray(eval(res.loc[1076, "polya_ratio"]), dtype="float32")
x_hat = np.asarray(eval(res.loc[1076, "hist_pred"]), dtype="float32")
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="rebuilt")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio", ylim=(0, 0.10))
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
x = np.asarray(eval(res.loc[75, "polya_ratio"]), dtype="float32")
x_hat = np.asarray(eval(res.loc[75, "hist_pred"]), dtype="float32")
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="rebuilt")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio", ylim=(0, 0.10))
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
edges = np.arange(10, 335, 5, dtype="float32")
centers = edges[:-1] + 2.5


def median_calculator(dist: list):
    p = np.asarray(dist, dtype="float32")
    cdf = np.cumsum(p)  # 累积分布 F(x_k)
    k = np.searchsorted(cdf, 0.5)  # 第一处 ≥0.5 的下标
    left_mass = cdf[k - 1] if k > 0 else 0.0
    gap = 0.5 - left_mass
    frac = gap / p[k]

    edge_left = centers[k] - 5 / 2
    median = edge_left + frac * 5
    return median


pred_med = res_med["hist_pred"].apply(eval).apply(median_calculator)
pred_med.head()

In [ ]:
def scatter_plot(x, y):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )

In [ ]:
scatter_plot(res_med["polya_median"], pred_med)

---
---

In [ ]:
cdna_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/ath_all_length_3utr_cdna.tsv",
    sep="\t",
)
cdna_df.head()

In [ ]:
exp_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/ath_expression.tsv",
    sep="\t",
)

# 4.Ath

In [ ]:
ath_seq = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/ath_all_length_3utr_cdna.tsv", sep="\t"
)
ath_exp = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/datasets/ath_expression.tsv", sep="\t"
)

mRNA	leaf_polya_lengths	3utr_sequence	cdna_sequence   leaf_CPM

In [ ]:
ath_leaf = ath_seq[
    ["mRNA", "leaf_polya_lengths", "3utr_sequence", "cdna_sequence"]
].merge(ath_exp[["mRNA", "leaf_CPM"]], on=["mRNA"])

In [ ]:
mask = ~pd.isna(ath_leaf["leaf_polya_lengths"])
ath_leaf = ath_leaf[mask].reset_index(drop=True)

In [ ]:
ath_leaf.head()

In [ ]:
count = ath_leaf["leaf_polya_lengths"].apply(lambda x: len(x.split(",")))
ath_leaf["count"] = count

In [ ]:
len(ath_leaf[ath_leaf["count"] >= 20])

In [ ]:
lengths = ath_leaf["leaf_polya_lengths"].str.split(",").explode().astype(float)
len_min = lengths.min()
len_max = lengths.max()
print(len_min, len_max)

In [ ]:
sub = lengths[lengths >= 170]
print(len(sub))
fig, ax = plt.subplots()
ax.hist(sub, bins=100, density=False, histtype="step")
ax.set_xlabel("polya_length")
ax.set_ylabel("count")

In [ ]:
def length_ratio(polya_str: str, bins=bins) -> str:
    lengths = np.fromstring(polya_str, sep=",")
    counts, _ = np.histogram(lengths, bins=bins)
    ratio = counts / counts.sum()
    return json.dumps(ratio.tolist())


bins = np.append(np.arange(10, 190, 20), np.inf)
ath_leaf["leaf_polya_ratio"] = ath_leaf["leaf_polya_lengths"].apply(length_ratio)

In [ ]:
ath_leaf.to_csv(
    "/DataZhai/zhangyt/project/ai/datasets/ath_leaf.tsv", sep="\t", index=False
)

In [ ]:
ath_leaf = pd.read_csv("/DataZhai/zhangyt/project/ai/datasets/ath_leaf.tsv", sep="\t")

In [ ]:
filtered_df = ath_leaf[
    (ath_leaf["polya_lengths_count"] >= 20)  # & (ath_leaf["length"] >= 2500)
].reset_index(drop=True)
y_df = filtered_df[["mRNA", "leaf_polya_lengths", "leaf_polya_ratio", "leaf_CPM"]]
x_encode_ls = []
for i in range(len(filtered_df)):
    seq = filtered_df.loc[i, "cdna_seq(gene+3utr)"].upper()
    if len(seq) < 3000:
        norm_seq = "".join(["N" * (3000 - len(seq)), seq])
    else:
        norm_seq = seq[-3000:]
    x_encode_ls.append(encode_seq(norm_seq))
x_ary = np.stack(x_encode_ls, axis=0)

In [ ]:
filtered_df.to_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/ath/ath_leaf_20_counts.tsv",
    sep="\t",
    index=False,
)
np.save("/DataZhai/zhangyt/project/ai/polya_dist/ath/x_input.npy", x_ary)
y_df.to_csv("/DataZhai/zhangyt/project/ai/polya_dist/ath/y_input.csv", index=False)

In [ ]:
x_ary = np.load("/DataZhai/zhangyt/project/ai/polya_dist/ath/x_input.npy")
y_df = pd.read_csv("/DataZhai/zhangyt/project/ai/polya_dist/ath/y_input.csv")

### Cross-entropy with only seq

In [ ]:
res = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/ath/crossentropy_train_val_CV_result.tsv",
    sep="\t",
)

In [ ]:
x = np.asarray(eval(res.loc[289, "polya_ratio"]), dtype="float32")
x_hat = np.asarray(eval(res.loc[289, "hist_pred"]), dtype="float32")
edges = np.arange(10, 210, 20, dtype="float32")
centers = edges[:-1] + 10

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="predicted")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio")
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
y_df = pd.read_csv("/DataZhai/zhangyt/project/ai/polya_dist/ath/y_input.csv")
res_med = res.merge(y_df[["mRNA", "leaf_polya_lengths"]], on=["mRNA"])

In [ ]:
edges = np.arange(10, 210, 20, dtype="float32")
centers = edges[:-1] + 10


def median_calculator(dist: list):
    p = np.asarray(dist, dtype="float32")
    cdf = np.cumsum(p)  # 累积分布 F(x_k)
    k = np.searchsorted(cdf, 0.5)  # 第一处 ≥0.5 的下标
    left_mass = cdf[k - 1] if k > 0 else 0.0
    gap = 0.5 - left_mass
    frac = gap / p[k]

    edge_left = centers[k] - 10
    median = edge_left + frac * 20
    return median


pred_med = res_med["hist_pred"].apply(eval).apply(median_calculator)
true_med = (
    res_med["leaf_polya_lengths"]
    .apply(lambda x: [eval(i) for i in x.split(",")])
    .apply(np.median)
)
print(pred_med.head())
print(true_med.head())

In [ ]:
def scatter_plot(x, y):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )

In [ ]:
scatter_plot(true_med, pred_med)

### Cross-entropy with seq & CPM

In [ ]:
res_cpm = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/polya_dist/ath/crossentropy_with_cpm_train_val_CV_result.tsv",
    sep="\t",
)

In [ ]:
x = np.asarray(eval(res_cpm.loc[289, "polya_ratio"]), dtype="float32")
x_hat = np.asarray(eval(res_cpm.loc[289, "hist_pred"]), dtype="float32")
edges = np.arange(10, 210, 20, dtype="float32")
centers = edges[:-1] + 10

kl = tf.reduce_sum(x * (tf.math.log(x + 1e-8) - tf.math.log(x_hat + 1e-8))).numpy()
w2 = np.sqrt(ot.wasserstein_1d(centers, centers, x, x_hat, p=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.stairs(x, edges, label="measured")
ax.stairs(x_hat, edges, label="predicted")

ax.set(xlabel="poly(A) length (nt)", ylabel="ratio")
ax.legend()
ax.text(
    0.95,
    0.92,
    f"KL={kl:.3f}\nW₂={w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.show()

In [ ]:
y_df = pd.read_csv("/DataZhai/zhangyt/project/ai/polya_dist/ath/y_input.csv")
res_med = res.merge(y_df[["mRNA", "leaf_polya_lengths"]], on=["mRNA"])

In [ ]:
edges = np.arange(10, 210, 20, dtype="float32")
centers = edges[:-1] + 10


def median_calculator(dist: list):
    p = np.asarray(dist, dtype="float32")
    cdf = np.cumsum(p)  # 累积分布 F(x_k)
    k = np.searchsorted(cdf, 0.5)  # 第一处 ≥0.5 的下标
    left_mass = cdf[k - 1] if k > 0 else 0.0
    gap = 0.5 - left_mass
    frac = gap / p[k]

    edge_left = centers[k] - 10
    median = edge_left + frac * 20
    return median


pred_med = res_med["hist_pred"].apply(eval).apply(median_calculator)
true_med = (
    res_med["leaf_polya_lengths"]
    .apply(lambda x: [eval(i) for i in x.split(",")])
    .apply(np.median)
)
print(pred_med.head())
print(true_med.head())

In [ ]:
def scatter_plot(x, y):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )

In [ ]:
scatter_plot(true_med, pred_med)

# 5.Single read

In [ ]:
ara_leaf_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/ara_leaf_polya_readseq_5per.tsv",
    sep="\t",
)[["read_id", "chr", "polya_length", "read_seq"]]

In [ ]:
ara_leaf_df["seq_len"] = ara_leaf_df["read_seq"].apply(str).apply(len)

In [ ]:
ara_leaf_df = ara_leaf_df[ara_leaf_df["seq_len"] > 3]

In [ ]:
ara_leaf_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/ara_leaf_reads_clean.tsv",
    sep="\t",
    index=False,
)

In [ ]:
len(ara_leaf_df[(ara_leaf_df["seq_len"] == 1500)])

In [ ]:
ara_leaf_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/ara_leaf_reads_clean.tsv",
    sep="\t",
)

In [ ]:
filtered_df = ara_leaf_df[
    (ara_leaf_df["polya_length"] <= 200) & (ara_leaf_df["seq_len"] <= 3000)
].reset_index(drop=True)

In [ ]:
filtered_df

In [ ]:
for chr, reads in filtered_df.groupby("chr"):
    print(chr, len(reads))

In [ ]:
ara_train_df = filtered_df[
    (filtered_df["chr"] == "chr1")
    | (filtered_df["chr"] == "chr2")
    | (filtered_df["chr"] == "chr3")
    | (filtered_df["chr"] == "chr5")
].reset_index(drop=True)
ara_test_df = filtered_df[(filtered_df["chr"] == "chr4")].reset_index(drop=True)

In [ ]:
ara_train_df

In [ ]:
y_df = ara_train_df[["read_id", "chr", "polya_length"]]
x_encode_ls = []
for i in range(len(ara_train_df)):
    seq = ara_train_df.loc[i, "read_seq"].upper()
    if len(seq) < 1500:
        norm_seq = "".join(["N" * (1500 - len(seq)), seq])
    else:
        norm_seq = seq[-1500:]
    x_encode_ls.append(encode_seq(norm_seq))
x_ary = np.stack(x_encode_ls, axis=0)

In [ ]:
y_df = ara_train_df[["read_id", "chr", "read_seq", "polya_length"]]
y_df["log1p"] = y_df["polya_length"].apply(np.log1p)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(y_df["log1p"].to_numpy().reshape(-1, 1))
y_scaled = scaler.transform(y_df["log1p"].to_numpy().reshape(-1, 1))  # 均值 0、方差 1

In [ ]:
y_df["log1p_scaled"] = y_scaled

In [ ]:
y_df

In [ ]:
y_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/ara_leaf_3000_chr1235.csv",
    index=False,
)

In [ ]:
ara_train_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/ath_leaf_train_800_1500.tsv",
    sep="\t",
    index=False,
)
np.save("/DataZhai/zhangyt/project/ai/single_read/ara_leaf/x_input.npy", x_ary)
y_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/y_input.csv", index=False
)

## gene_200

In [ ]:
train_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/train_data_leaf_shuffled.250820.csv"
)
eval_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/eval_data_leaf_shuffled.250820.csv"
)
test_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/test_data_leaf_shuffled.250820.csv"
)

In [ ]:
test_df

In [ ]:
ara_leaf_df = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/ara_leaf_polya_readseq_5per.tsv",
    sep="\t",
)

In [ ]:
y_log1p = test_df["label"].apply(np.log1p)
scaler = StandardScaler().fit(y_log1p.to_numpy().reshape(-1, 1))
joblib.dump(
    scaler,
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/test_scaler.pkl",
)
y_scaled = scaler.transform(y_log1p.to_numpy().reshape(-1, 1))
test_df["log1p_scaled"] = y_scaled.apply(lambda x: x + 2.3)

In [ ]:
eval_df["seq_len"] = eval_df["sequence"].apply(len)

In [ ]:
train_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/train_data_leaf.250820.csv",
    index=False,
)
eval_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/eval_data_leaf.250820.csv",
    index=False,
)
test_df.to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/test_data_leaf.250820.csv",
    index=False,
)

In [ ]:
train_df.sample(frac=1).reset_index(drop=True).to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/train_data_leaf_shuffled.250820.csv",
    index=False,
)
eval_df.sample(frac=1).reset_index(drop=True).to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/eval_data_leaf_shuffled.250820.csv",
    index=False,
)
test_df.sample(frac=1).reset_index(drop=True).to_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/test_data_leaf_shuffled.250820.csv",
    index=False,
)

In [ ]:
# 读取
scaler = joblib.load("label_scaler.pkl")

# 假设有标准化后的 y_scaled
y_log = scaler.inverse_transform(y_scaled.reshape(-1, 1)).ravel()  # 还原到 log1p 空间
y_raw = np.expm1(y_log)

In [ ]:
x_encode_ls = []
for i in range(len(test_df)):
    seq = test_df.loc[i, "sequence"].upper()
    if len(seq) < 1800:
        norm_seq = "".join(["N" * (1800 - len(seq)), seq])
    else:
        norm_seq = seq[-1800:]
    x_encode_ls.append(encode_seq(norm_seq))
x_ary = np.stack(x_encode_ls, axis=0)
np.save(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/test_data_leaf.250820.npy",
    x_ary,
)

In [ ]:
res = pd.read_csv(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/ath_gt_gene200+_predictions.csv"
)
scaler = joblib.load(
    "/DataZhai/zhangyt/project/ai/single_read/ara_leaf/gene_200/eval_scaler.pkl"
)
y_log1p = res["y_true"].apply(np.log1p)
y_scaled = scaler.transform(y_log1p.to_numpy().reshape(-1, 1))
measured = (y_scaled + 2.3).ravel()
pred = res["y_pred"]

In [ ]:
print(measured)

In [ ]:
def scatter_plot(x, y):
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values)(values)
    spearman_corr, _ = stats.spearmanr(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    n = len(x)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=kernel, cmap="RdYlBu_r", s=20)
    plt.xlabel("Measured data")
    plt.ylabel("Predicted data")
    plt.colorbar(scatter, label="Density")
    plt.axis("equal")
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val], "--", color="grey", linewidth=1)
    plt.text(
        0.95,
        0.05,
        f"n={n}\nPearson: {pearson_corr:.2f}\nSpearman: {spearman_corr:.2f}",
        horizontalalignment="right",
        verticalalignment="bottom",
        transform=plt.gca().transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.5),
    )

In [ ]:
scatter_plot(measured, pred)